In [ ]:
import os
import json
import glob
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy.interpolate import interp1d
import warnings

# Suppress harmless warnings from undefined correlations when driving perfectly straight
warnings.filterwarnings("ignore")

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
ZIP_PATH = "synchronized_output.zip"
EXTRACT_DIR = "synchronized_output"

if os.path.exists(ZIP_PATH) and not os.path.exists(EXTRACT_DIR):
    print("Extracting Test_data.zip...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(".")

# ==========================================
# 2. CONTINUOUS 1D RTS KALMAN SMOOTHER
# ==========================================
class KalmanSmoother1D:
    def __init__(self, dt, q_var, q_bias, r_var):
        self.dt = dt
        self.F = np.array([[1.0, -dt], [0.0, 1.0]])
        self.B = np.array([dt, 0.0])
        self.H = np.array([[1.0, 0.0]])
        self.Q = np.array([[q_var, 0], [0, q_bias]])
        self.R_base = r_var

    def smooth(self, u, z, z_mask, initial_state, v_ref=None, is_heading=False):
        N = len(u)
        x_fwd = np.zeros((N, 2))
        P_fwd = np.zeros((N, 2, 2))
        x_pred = np.zeros((N, 2))
        P_pred = np.zeros((N, 2, 2))

        # Initialize [State, Bias]
        x_fwd[0] = [initial_state, 0.0]
        P_fwd[0] = np.eye(2) * 5.0

        # --- FORWARD PASS (EKF) ---
        for i in range(1, N):
            x_p = self.F @ x_fwd[i-1] + self.B * u[i-1]
            P_p = self.F @ P_fwd[i-1] @ self.F.T + self.Q

            x_pred[i], P_pred[i] = x_p.copy(), P_p.copy()

            if z_mask[i]:
                R = self.R_base
                # Dynamic Covariance
                if is_heading and v_ref is not None:
                    speed = max(v_ref[i], 0.1)
                    if speed < 2.5:
                        R += 5.0 / (speed ** 2)

                y = z[i] - (self.H @ x_p)[0]
                S = self.H @ P_p @ self.H.T + R
                K = P_p @ self.H.T @ np.linalg.inv(S)

                x_fwd[i] = x_p + (K * y).flatten()
                P_fwd[i] = (np.eye(2) - K @ self.H) @ P_p
            else:
                x_fwd[i] = x_p
                P_fwd[i] = P_p

        # --- BACKWARD PASS (RTS Smoother) ---
        x_smooth = np.zeros((N, 2))
        P_smooth = np.zeros((N, 2, 2))
        x_smooth[-1] = x_fwd[-1]
        P_smooth[-1] = P_fwd[-1]

        for i in range(N-2, -1, -1):
            C = P_fwd[i] @ self.F.T @ np.linalg.pinv(P_pred[i+1])
            x_smooth[i] = x_fwd[i] + C @ (x_smooth[i+1] - x_pred[i+1])
            P_smooth[i] = P_fwd[i] + C @ (P_smooth[i+1] - P_pred[i+1]) @ C.T

        return x_smooth

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def align_axis(df_sensor, t_gps, ref_diff, default_ax='z', default_sign=1.0):
    if len(t_gps) < 2: return default_ax, default_sign, 0.0

    t_sens = df_sensor['seconds_elapsed'].values
    best_corr = -1.0
    best_ax = default_ax
    best_sign = default_sign

    for ax in ['x', 'y', 'z']:
        sig = df_sensor[ax].values
        # Safe flat-line extrapolation prevents explosive spirals at boundaries
        f_ref = interp1d(t_gps, ref_diff, bounds_error=False, fill_value=(ref_diff[0], ref_diff[-1]))(t_sens)

        if np.std(sig) > 1e-4 and np.std(f_ref) > 1e-4:
            corr = np.corrcoef(sig, f_ref)[0, 1]
            if not np.isnan(corr) and abs(corr) > best_corr:
                best_corr = abs(corr)
                best_ax = ax
                best_sign = np.sign(corr) if corr != 0 else 1.0

    # Confidence Fallback: If driving perfectly straight, disable IMU noise injection
    if best_corr < 0.25:
        return default_ax, default_sign, 0.0
    return best_ax, best_sign, 1.0

# ==========================================
# 4. MAIN PROCESSING PIPELINE
# ==========================================
def process_frame(folder_path):
    print(f"Attempting to process folder: {os.path.basename(folder_path)}")
    # Modify to specifically look for 'sensor_data.json'
    json_file_path = os.path.join(folder_path, "sensor_data.json")
    img_files = glob.glob(os.path.join(folder_path, "*.jpg")) + glob.glob(os.path.join(folder_path, "*.png"))

    if not os.path.exists(json_file_path):
        print(f"Skipping {os.path.basename(folder_path)}: 'sensor_data.json' not found.")
        return

    print(f"Opening JSON file: {os.path.basename(json_file_path)}")

    with open(json_file_path, 'r') as f:
        data = json.load(f)

    print(f"Keys in loaded JSON data: {data.keys()}")

    df_imu = pd.DataFrame(data.get('imu', []))
    df_gyr = pd.DataFrame(data.get('gyro', []))
    df_gps = pd.DataFrame(data.get('gps', []))

    if len(df_gps) < 2 or len(df_imu) < 2 or len(df_gyr) < 2:
        print(f"Skipping {os.path.basename(folder_path)}: Insufficient sensor data (GPS: {len(df_gps)}, IMU: {len(df_imu)}, Gyro: {len(df_gyr)}).")
        return

    dt = 0.02
    t_start = data.get('frame_timestamp_sec', df_imu['seconds_elapsed'].min())
    time_window = data.get('time_window_sec', 5.0)

    N = int(round(time_window / dt)) + 1
    t_eval = np.linspace(t_start, t_start + time_window, N)

    t_gps = df_gps['seconds_elapsed'].values
    v_gps = df_gps['speed'].values.astype(float)
    raw_bearing = df_gps['bearing'].values.astype(float)

    # -----------------------------------------------------
    # BUG FIX 1: GPS STATIONARY SINGULARITY SANITIZATION
    # -----------------------------------------------------
    # When speed drops below 1.0 m/s, GPS bearing artificially resets to 0 (North).
    # We strip these invalid artifacts and interpolate the real heading over the gap.
    if 'bearingAccuracy' in df_gps.columns:
        valid_b_mask = (v_gps >= 1.0) & (df_gps['bearingAccuracy'].values > 0.0)
    else:
        valid_b_mask = v_gps >= 1.0

    if np.sum(valid_b_mask) >= 2:
        t_valid = t_gps[valid_b_mask]
        b_valid = raw_bearing[valid_b_mask]
        # ROS FLU Math Angle: East=0, North=pi/2 (+Yaw = Left Turn)
        math_b_valid = np.unwrap(np.deg2rad(90.0 - b_valid))
        clean_math_bearing = np.interp(t_gps, t_valid, math_b_valid)
    elif np.sum(valid_b_mask) == 1:
        clean_math_bearing = np.full(len(t_gps), np.deg2rad(90.0 - raw_bearing[valid_b_mask][0]))
    else:
        clean_math_bearing = np.full(len(t_gps), np.deg2rad(90.0 - raw_bearing[0]))

    # 1. Dynamic Sensor Alignment
    gps_v_diff = np.gradient(v_gps, t_gps)
    gps_w_diff = np.gradient(clean_math_bearing, t_gps)

    ax_fwd, sign_fwd, conf_a = align_axis(df_imu, t_gps, gps_v_diff, default_ax='y', default_sign=1.0)
    ax_yaw, sign_yaw, conf_w = align_axis(df_gyr, t_gps, gps_w_diff, default_ax='z', default_sign=1.0)

    a_raw = df_imu[ax_fwd].values * sign_fwd
    w_raw = df_gyr[ax_yaw].values * sign_yaw

    # 2. Linear Regression DC-Bias & Gravity Leakage Removal
    v_ref = interp1d(t_gps, v_gps, bounds_error=False, fill_value=(v_gps[0], v_gps[-1]))(t_eval)
    w_ref = interp1d(t_gps, clean_math_bearing, bounds_error=False, fill_value=(clean_math_bearing[0], clean_math_bearing[-1]))(t_eval)

    mean_gps_a = (v_ref[-1] - v_ref[0]) / time_window
    mean_gps_w = (w_ref[-1] - w_ref[0]) / time_window

    a_adj = a_raw - np.mean(a_raw) + mean_gps_a
    w_adj = w_raw - np.mean(w_raw) + mean_gps_w

    # Synthetic Control injection if Confidence drops (protects against unalignable traffic jams)
    a_eval = interp1d(df_imu['seconds_elapsed'].values, a_adj, bounds_error=False, fill_value=(a_adj[0], a_adj[-1]))(t_eval)
    w_eval = interp1d(df_gyr['seconds_elapsed'].values, w_adj, bounds_error=False, fill_value=(w_adj[0], w_adj[-1]))(t_eval)

    u_a = a_eval * conf_a + np.gradient(v_ref, dt) * (1.0 - conf_a)
    u_w = w_eval * conf_w + np.gradient(w_ref, dt) * (1.0 - conf_w)

    # 3. Prepare GPS Anchor Points
    z_mask_v = np.zeros(N, dtype=bool)
    z_mask_w = np.zeros(N, dtype=bool)
    z_v, z_w = np.zeros(N), np.zeros(N)

    for i, t_g in enumerate(t_gps):
        if t_g < t_eval[0] - dt or t_g > t_eval[-1] + dt: continue
        idx = np.argmin(np.abs(t_eval - t_g))
        if np.abs(t_eval[idx] - t_g) <= dt:
            z_mask_v[idx] = True
            z_v[idx] = v_gps[i]

            # Mask out toxic GPS heading anchors during stops (Lets gyro strictly coast the intersection)
            if valid_b_mask[i]:
                z_mask_w[idx] = True
                z_w[idx] = clean_math_bearing[i]

    # Force Anchors at perfectly exact boundaries to lock smoother drift to exactly 0.0
    z_mask_v[0], z_mask_v[-1] = True, True
    z_mask_w[0], z_mask_w[-1] = True, True
    z_v[0], z_v[-1] = v_ref[0], v_ref[-1]
    z_w[0], z_w[-1] = w_ref[0], w_ref[-1]

    # 4. Apply Continuous RTS Smoother
    # q_bias=0.1 allows the bias tracker to instantly react to heavy braking suspension pitch
    vel_smoother = KalmanSmoother1D(dt, q_var=0.5, q_bias=0.1, r_var=1.0)
    out_v = vel_smoother.smooth(u_a, z_v, z_mask_v, initial_state=v_ref[0])

    yaw_smoother = KalmanSmoother1D(dt, q_var=0.05, q_bias=1e-4, r_var=0.5)
    out_w = yaw_smoother.smooth(u_w, z_w, z_mask_w, initial_state=w_ref[0], v_ref=v_ref, is_heading=True)

    # 5. Integrate Kinematics (ROS ISO-8855 Frame: X=Forward, Y=Left)
    v_smooth = np.clip(out_v[:, 0], 0.0, None)
    yaw_smooth = out_w[:, 0]
    yaw_rate_smooth = u_w - out_w[:, 1]

    # Localize relative to start
    yaw_local = yaw_smooth - yaw_smooth[0]
    yaw_local_wrapped = (yaw_local + np.pi) % (2 * np.pi) - np.pi

    # +Yaw = Left Turn -> +Y displacement
    vx = v_smooth * np.cos(yaw_local)
    vy = v_smooth * np.sin(yaw_local)

    x, y = np.zeros(N), np.zeros(N)
    for i in range(1, N):
        x[i] = x[i-1] + 0.5 * (vx[i] + vx[i-1]) * dt
        y[i] = y[i-1] + 0.5 * (vy[i] + vy[i-1]) * dt

    # 6. Output JSON (Shifted back to Relative Time for graphing ease)
    t_relative = t_eval - t_start
    trajectory_list = []

    for i in range(N):
        trajectory_list.append({
            "time_sec": float(round(t_relative[i], 3)),
            "x": float(round(x[i], 4)),
            "y": float(round(y[i], 4)),
            "vx": float(round(vx[i], 4)),
            "vy": float(round(vy[i], 4)),
            "yaw": float(round(yaw_local_wrapped[i], 4)),
            "yaw_rate": float(round(yaw_rate_smooth[i], 4))
        })

    with open(os.path.join(folder_path, "trajectory.json"), 'w') as f:
        json.dump(trajectory_list, f, indent=4)

    # ==========================================
    # 7. VALIDATION VISUALIZATION
    # ==========================================
    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(f"Trajectory Validation: {os.path.basename(folder_path)}", fontsize=16)

    ax1 = plt.subplot(1, 3, 1)
    if img_files: ax1.imshow(mpimg.imread(img_files[0]))
    ax1.set_title("Camera Frame")
    ax1.axis('off')

    ax2 = plt.subplot(1, 3, 2)
    ax2_twin = ax2.twinx()
    ax2.plot(t_relative, v_smooth, 'b-', label='Smoothed Speed', linewidth=2)

    valid_time_mask = (t_gps >= t_start) & (t_gps <= t_start + time_window)
    ax2.plot(t_gps[valid_time_mask] - t_start, v_gps[valid_time_mask], 'ro', label='GPS Speed')

    ax2_twin.plot(t_relative, np.rad2deg(yaw_local_wrapped), 'g-', label='Local Yaw')

    # Plot sanitized GPS anchors showing exactly which data was trusted
    valid_w_plot = valid_time_mask & valid_b_mask
    gps_yaw_local = clean_math_bearing[valid_w_plot] - yaw_smooth[0]
    gps_yaw_local_wrapped = (gps_yaw_local + np.pi) % (2 * np.pi) - np.pi
    ax2_twin.plot(t_gps[valid_w_plot] - t_start, np.rad2deg(gps_yaw_local_wrapped), 'mo', alpha=0.6, label='Trusted GPS Yaw')

    ax2.set_xlabel('Relative Time (sec)')
    ax2.set_ylabel('Velocity (m/s)', color='b')
    ax2_twin.set_ylabel('Yaw (deg) [+ is Left Turn]', color='g')
    ax2.set_title(f"Fusion | Align: ({ax_fwd}, {ax_yaw}) | Conf: ({conf_a:.1f}, {conf_w:.1f})")
    ax2.grid(True, linestyle='--', alpha=0.6)

    ax3 = plt.subplot(1, 3, 3)
    ax3.set_aspect('equal', adjustable='datalim')

    ax3.plot(y, x, 'b.-', markersize=3, label='Vehicle Path')
    ax3.plot(0, 0, 'go', markersize=10, label='Start (0,0)')
    ax3.plot(y[-1], x[-1], 'rs', markersize=8, label='End')
    ax3.axvline(0, color='k', linestyle='--', alpha=0.5)

    # --- BUG FIX 2: Matplotlib Axis Mirror ---
    # By default, Matplotlib plots negative numbers on the Left side.
    # In ROS FLU Math, a Right Turn yields a Negative Y.
    # We must set positive limits on the Left, and negative on the Right to match reality!
    max_disp = max(np.max(np.abs(y)) * 1.5, 5.0)
    ax3.set_xlim(max_disp, -max_disp)
    ax3.set_ylim(min(np.min(x)-2, -2), np.max(x) + 5)

    ax3.set_xlabel('Lateral Displacement (Y) [m]\n<-- Left Turn (+Y)   |   Right Turn (-Y) -->')
    ax3.set_ylabel('Forward Displacement (X) [m]')
    ax3.set_title("Bird's-Eye View (ROS ISO-8855 Standard)")
    ax3.legend()
    ax3.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.savefig(os.path.join(folder_path, "validation_plot.png"), dpi=120)
    plt.close()
    print(f"Success: {os.path.basename(folder_path)} -> Rendered successfully.")

# ==========================================
# EXECUTION TRIGGER
# ==========================================
if os.path.exists(EXTRACT_DIR):
    frame_folders = [os.path.dirname(f) for f in glob.glob(os.path.join(EXTRACT_DIR, "**", "sensor_data.json"), recursive=True)]
    if not frame_folders:
        frame_folders = [f for f in glob.glob(os.path.join(EXTRACT_DIR, '*')) if os.path.isdir(f)]

    print(f"Discovered {len(frame_folders)} frames. Processing...")
    for folder in sorted(list(set(frame_folders))):
        try:
            process_frame(folder)
        except Exception as e:
            print(f"Error processing {folder}: {e}")
else:
    print("Data directory not found. Upload Test_data.zip to Colab.")

Discovered 2247 frames. Processing...
Attempting to process folder: 2025-08-11T16-36-38+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


Success: 2025-08-11T16-36-38+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-36-38.033289+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


Success: 2025-08-11T16-36-38.033289+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-36-38.066611+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


Success: 2025-08-11T16-36-38.066611+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-36-38.099956+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


Success: 2025-08-11T16-36-38.099956+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-36-38.133267+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


Success: 2025-08-11T16-36-38.133267+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-36-38.166578+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


... (851 outputs left out)

Success: 2025-08-11T16-37-06.527844+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-37-06.561056+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


Success: 2025-08-11T16-37-06.561056+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-37-06.594367+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


Success: 2025-08-11T16-37-06.594367+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-37-06.627822+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])


Success: 2025-08-11T16-37-06.627822+00-00 -> Rendered successfully.
Attempting to process folder: 2025-08-11T16-37-06.661089+00-00
Opening JSON file: sensor_data.json
Keys in loaded JSON data: dict_keys(['frame_timestamp_utc', 'time_window_sec', 'imu', 'gyro', 'gps'])
